# Clean up FNO pipeline

This is for the 2D Darcy problem on [0, 1]^2, with input permeability field discrete in {0, 1}.

### Try to improve the MCMC step to bridge the gap between 2.4 and 24 NLL

In [ ]:
# Imports
import numpy as np
import sys

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern

import torch
from torch.utils.data import DataLoader
import neuralop
from neuralop.models import FNO
from neuralop import Trainer, LpLoss, H1Loss
from neuralop.training import AdamW
from neuralop.data.datasets import load_darcy_flow_small, DarcyDataset
from neuralop.utils import count_model_params
from torch.nn import functional as F

import matplotlib.pyplot as plt


In [ ]:
# ===== HYPERPARAMETERS =====

# === Data ===
data_n_train = 1_000
data_n_tests = [100, 50]
data_train_resolution = 64  # options: 16, 32, 64, 128, 421
data_test_resolutions = [64, 128]
data_batch_size = 32
data_test_batch_sizes = [32, 32]

# === Model ===
FNO_n_modes=(12, 12)  # these are the params which are saved. later change to (25,25). anyhow will have to vary for ablation study
FNO_in_channels=1
FNO_out_channels=1
FNO_hidden_channels=32  # Width of model. Documentation recommends 64, but 32 is sufficient for our purposes and much faster to train.
FNO_n_layers = 4  # Number of Fourier layers, the default
FNO_projection_channel_ratio=2
FNO_factorization="tucker"
FNO_rank=0.42
FNO_nonlinearity= F.gelu

# === Training ===
TRAIN_epochs = 300


In [ ]:
def T_push(x, tau=-1, boundary=0):
    """A smooth approximation to the step function, which can be used to get a binary field from the GP sample while maintaining differentiability.
    See Akiyldiz for details.
    Combined with the option to get a hard threshold by setting tau=-1, based on the usual 3-12 normalisation.
    """
    if isinstance(x, np.ndarray):
        if tau == -1:
            return np.where(x < boundary, 3.0, 12.0)
        return 0.5 * np.tanh(tau * x) + 0.5
    else:  # torch tensor
        if tau == -1:
            return torch.where(x < boundary,
                               torch.tensor(3.0),
                               torch.tensor(12.0))
        return 0.5 * torch.tanh(tau * x) + 0.5

In [ ]:
# Activate GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Import Data
# Loading the Darcy-Flow dataset

dataset = DarcyDataset(
    root_dir="./darcy_data",
    n_train=data_n_train,
    n_tests=data_n_tests,
    train_resolution=data_train_resolution,
    batch_size=data_batch_size,
    test_resolutions=data_test_resolutions,
    test_batch_sizes=data_test_batch_sizes,
)

# renormalise data to 3-12 range, as per convention. For some reason the default normalisation is 0-1 which is not the norm
# only run the code ONCE, otherwise it will normalise twice and all entries become 12.
for test_set in dataset.test_dbs.values():
    test_set.x = T_push(test_set.x, tau=-1, boundary=0.5)

dataset._train_db.x = T_push(dataset._train_db.x, tau=-1, boundary=0.5)


train_loader = DataLoader(
        dataset.train_db,
        batch_size=data_batch_size,
        num_workers=1,
        pin_memory=True,
        persistent_workers=False,
    )

test_loaders = {}
for res, test_bsize in zip(data_test_resolutions, data_test_batch_sizes):
    test_loaders[res] = DataLoader(
        dataset.test_dbs[res],
        batch_size=test_bsize,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        persistent_workers=False,
    )

data_processor = dataset.data_processor



In [ ]:
# Instantiate Model
model = FNO(
    n_modes=FNO_n_modes,
    in_channels=FNO_in_channels,
    out_channels=FNO_out_channels,
    hidden_channels=FNO_hidden_channels,
    n_layers=FNO_n_layers,
    # projection_channel_ratio=FNO_projection_channel_ratio,
    factorization=FNO_factorization,
    non_linearity=FNO_nonlinearity,
    # rank=FNO_rank,
)

# NOTE: FNO takes in inputs of shape (batch_size, in_channels, height, width) and outputs the same shape.
# If you want to pass a single observation, you need to add batch and channel dimensions, e.g., x[0, 0].unsqueeze(0).unsqueeze(0) to get shape (1, 1, 16, 16).

model = model.to(device)

n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()


# Scheduler and Loss

optimizer = AdamW(model.parameters(), lr=8e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

l2loss = LpLoss(d=2, p=2)
h1loss = H1Loss(d=2)

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}


In [ ]:
save_dir = "./checkpoints_FNO_Darcy"

# Train model
# Creating the trainer
trainer = Trainer(
    model=model,
    n_epochs=TRAIN_epochs,
    device=device,
    data_processor=data_processor,
    wandb_log=False,
    eval_interval=5,
    use_distributed=False,
    verbose=True,
)

torch.serialization.add_safe_globals([torch._C._nn.gelu])
torch.serialization.add_safe_globals([neuralop.layers.spectral_convolution.SpectralConv])

# DELETE checkpoints FOLDER BEFORE TRAINING FROM SCRATCH
# We train and save checkpoints
# Train from scratch
trainer.train(
    train_loader=train_loader,
    test_loaders={},
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    save_every=5,
    save_dir=save_dir,
    # resume_from_dir=save_dir,
)



# # Train from checkpoints
# trainer.train(
#     train_loader=train_loader,
#     test_loaders={},
#     optimizer=optimizer,
#     scheduler=scheduler,
#     regularizer=False,
#     training_loss=train_loss,
#     resume_from_dir="./checkpoints",
#     save_dir="./checkpoints",
# )

In [ ]:
# ===========================================================================
# MCMC CONFIGURATION
# ===========================================================================
# Toggle the proposal here. Everything else in the loop is shared so the two
# schemes are a clean A/B:
#   USE_GRADIENT_PROPOSAL = False  ->  vanilla pCN  (prior-reversible, exact)
#   USE_GRADIENT_PROPOSAL = True   ->  pCN-MALA     (gradient-informed drift)
USE_GRADIENT_PROPOSAL = True

# --- which observation to invert ---
MCMC_resolution = 64          # 64 = standard task, 128 = super-resolution task

# --- chain length / adaptation ---
MCMC_n_iterations = 80_000
MCMC_delta        = 0.00025   # step size (adapted during burn-in)
adapt_interval    = 500
adapt_cutoff      = 60_000
delta_min, delta_max = 1e-6, 0.49
burn_in           = 20_000    # samples before this are excluded from the posterior mean

# Optimal acceptance differs by scheme (RW/pCN ~0.3, Langevin/MALA ~0.574)
target_acceptance = 0.574 if USE_GRADIENT_PROPOSAL else 0.30

# --- gradient (pCN-MALA only): sharpness of the differentiable threshold used
#     ONLY to build the drift direction. The accept/reject test still uses the
#     exact hard-threshold {3,12} model, so both toggles target the same posterior.
MALA_tau = 5.0

# ===========================================================================
# Observation, true field, and sigma
# ===========================================================================
true_field        = dataset.test_dbs[MCMC_resolution].x[0, 0].unsqueeze(0).unsqueeze(0)
true_observations = dataset.test_dbs[MCMC_resolution].y[0, 0].unsqueeze(0).unsqueeze(0).to(device)

# Calibrate sigma to the FNO's OWN forward error at this resolution (the honest
# surrogate-noise level), not the observation std. This sets the NLL scale.
theta_true_tensor = true_field.to(device).float()
with torch.no_grad():
    _pred = model(theta_true_tensor)
    _pred = _pred * data_processor.out_normalizer.std + data_processor.out_normalizer.mean
sigma = (true_observations - _pred[0, 0]).std().item()
print(f"MCMC resolution: {MCMC_resolution} | proposal: "
      f"{'pCN-MALA' if USE_GRADIENT_PROPOSAL else 'vanilla pCN'}")
print(f"sigma (FNO forward-error std): {sigma:.4f}")
print(f"observation std (for reference): {true_observations.std().item():.4f}")


In [ ]:
# MCMC helper functions

def log_likelihood(x: torch.Tensor, y: torch.Tensor, model, sigma=0.5275) -> torch.Tensor:
    """Inputs:
    - x: input tensor (location measurements of field θ)
    - y: observed output tensor (e.g., solution field u at locations x)
    - model: the trained FNO model that maps x to predicted y
    - sigma: standard deviation of the Gaussian likelihood
    Returns:
    - log-likelihood of observing y given x under the model's predictions"""
    # Ensure x and y are 4D tensors with shape (1, 1, height, width)
    # Ensure both x and y are on the same device as the model
    # x = x.to(device)
    # y = y.to(device)
    # Assuming Gaussian likelihood with fixed variance
    with torch.no_grad():
        forwards = model(x)
        mean = data_processor.out_normalizer.mean  # need to unnormalize the model outputs before comparing to y, since the model was trained on normalized data
        std = data_processor.out_normalizer.std
        forwards = forwards * std + mean
    forwards = forwards[0, 0]  # need .cpu() before .numpy()
    ll = -0.5 * torch.sum((y - forwards) ** 2).item() / sigma**2
    return ll


# Before the loop, build once:
_x_coords = np.linspace(0, 1, data_train_resolution)
_y_coords = np.linspace(0, 1, data_train_resolution)
_X, _Y = np.meshgrid(_x_coords, _y_coords)
_grid_points = np.column_stack([_X.ravel(), _Y.ravel()])
_gp = GaussianProcessRegressor(kernel=RBF(length_scale=0.3))

def sample_GP():
    sample = _gp.sample_y(_grid_points, n_samples=1, random_state=None)
    return sample.reshape(data_train_resolution, data_train_resolution)


# Before the loop
from sklearn.gaussian_process.kernels import Matern
# _kernel = RBF(length_scale=0.25)
_kernel = Matern(length_scale=0.3, nu=1) # nu=0.5 is even rougher
K_1d = _kernel(_x_coords.reshape(-1, 1))
K_1d += 1e-6 * np.eye(len(K_1d))
L_1d = np.linalg.cholesky(K_1d)


def sample_GP_kronecker():
    Z = np.random.randn(data_train_resolution, data_train_resolution)
    return L_1d @ Z @ L_1d.T


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 64    # physical grid resolution (unchanged)
M = 256   # oversampling factor — controls KL truncation level (M^2 terms)
L = 1.0


def sample_GP_operator(N=64, M=256):
    # Fine-grid wavenumbers — M^2 KL terms instead of N^2
    h_fine = 1 / M  # fine grid spacing for Fourier sampling
    kx = 2 * np.pi * np.fft.fftfreq(M, d=h_fine)
    ky = 2 * np.pi * np.fft.fftfreq(M, d=h_fine)

    KX, KY = np.meshgrid(kx, ky, indexing='ij')

    # Eigenvalues on fine Fourier grid
    A_eigs = KX**2 + KY**2 + 9.0
    Cov_eigs = 1.0 / (A_eigs**2)

    # Sample M^2 independent KL coefficients
    z = np.random.randn(M, M) + 1j * np.random.randn(M, M)
    u_hat = np.sqrt(Cov_eigs) * z

    # IFFT on fine grid
    u_fine = np.fft.ifft2(u_hat).real * M**2  # shape (M, M)

    # Subsample back to coarse N x N grid
    step = M // N
    u_coarse = u_fine[::step, ::step]  # shape (N, N)
    
    return u_coarse

from scipy.ndimage import gaussian_filter

def sample_GP_operator_masked(u_current, band_frac=0.25, smooth_sigma=2.0, N=64):
    xi = sample_GP_operator(N=N)
    # percentile threshold -> ~band_frac of cells are "interface", robust to u's scale
    thresh = np.percentile(np.abs(u_current), band_frac * 100)
    mask = (np.abs(u_current) < thresh).astype(float)
    mask = gaussian_filter(mask, sigma=smooth_sigma, mode='constant', cval=0.0)
    mask = mask / (mask.max() + 1e-8)          # keep mask in [0, 1]
    return xi * mask

def sample_GP_operator_mixed(u_current, band_frac=0.25, smooth_sigma=2.0,
                             interface_weight=0.7, N=64):
    if np.random.rand() < interface_weight:
        return sample_GP_operator_masked(u_current, band_frac, smooth_sigma, N=N)
    return sample_GP_operator(N=N)            # full global pCN move

# visualize
sample = sample_GP_operator()
sample = sample_GP_operator_masked(sample)
print(type(sample))
plt.imshow(sample, extent=[0,1,0,1], origin='lower')
plt.colorbar()
plt.title(r'Sample from GP with covariance $(-\Delta + 9I)^{-2}$')
plt.show()

In [ ]:
# FINAL REPORT: visualize the sample and its pushforward under T, to show how the smooth thresholding works

tf1 = sample_GP_operator(M=256)
tf2 = T_push(tf1, tau=-1)

fig, axs = plt.subplots(1, 2, figsize=(8, 4), dpi=300)

for ax, tf, label in zip(axs, [tf1, tf2], ["Gaussian Sample", "Pushforward of the sample"]):
    ax.set_title(label, fontsize=11, loc='center', pad=6)
    im = ax.imshow(tf, extent=[0, 1, 0, 1], origin='lower',
                   vmin=tf.min(), vmax=tf.max())  # each plot autoscales to its own data
    ax.tick_params(labelsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
# plt.savefig('figures_for_report/sample_coeff_and_pushforward.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ===========================================================================
# MCMC building blocks: spectral GP prior, forward potential, gradient, proposals
# ===========================================================================
model.eval()

# --- output normaliser (matches log_likelihood from the cell above) ---
_out_mean = data_processor.out_normalizer.mean
_out_std  = data_processor.out_normalizer.std

# --- spectral GP prior: covariance C = (-Delta + 9I)^{-2} ---
# Diagonal in Fourier, so sampling, applying C, and the C^{-1} quadratic form
# (needed for the MALA acceptance) are all cheap and mutually consistent.
def make_gp_prior(N, alpha=9.0, target_std=None, n_cal=300):
    k = 2*np.pi*np.fft.fftfreq(N, d=1.0/N)
    KX, KY = np.meshgrid(k, k, indexing='ij')
    lam0 = 1.0/(KX**2 + KY**2 + alpha)**2          # C eigenvalues (power spectrum)
    def _base():
        z = np.random.randn(N, N)
        return np.fft.ifft2(np.sqrt(lam0)*np.fft.fft2(z, norm='ortho'), norm='ortho').real
    scale = 1.0
    if target_std is not None:                      # match the magnitude of sample_GP_operator
        s = np.mean([_base().std() for _ in range(n_cal)])
        scale = target_std/(s + 1e-12)
    lam = (scale**2)*lam0
    sqrt_lam, cinv = np.sqrt(lam), 1.0/lam
    def sample():                                   # u ~ N(0, C)  (real, C^{1/2} on white noise)
        z = np.random.randn(N, N)
        return np.fft.ifft2(sqrt_lam*np.fft.fft2(z, norm='ortho'), norm='ortho').real
    def cinv_quad(v):                               # <v, C^{-1} v>
        vh = np.fft.fft2(v, norm='ortho')
        return float(np.sum(cinv*np.abs(vh)**2))
    def apply_C(v):                                 # C v  (preconditions the gradient)
        vh = np.fft.fft2(v, norm='ortho')
        return np.fft.ifft2(lam*vh, norm='ortho').real
    return sample, cinv_quad, apply_C


# --- forward potential Phi(theta) = -log_likelihood = NLL (hard {3,12} model) ---
def Phi_hard(u, y, sigma):
    theta = T_push(u, tau=-1)                       # numpy {3,12}
    tt = torch.tensor(theta, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
    return -log_likelihood(tt, y, model, sigma=sigma)   # reuse the exact baseline forward


# --- differentiable surrogate potential + its gradient wrt the latent u ---
# Uses a SOFT threshold so d theta/du != 0; this only steers the proposal.
def Phi_soft_and_grad(u, y, sigma, tau=MALA_tau):
    u_t = torch.tensor(u, dtype=torch.float32, device=device, requires_grad=True)
    theta = 7.5 + 4.5*torch.tanh(tau*u_t)           # soft map to {3,12}: ->3 (u<0), ->12 (u>0)
    pred = model(theta.unsqueeze(0).unsqueeze(0))
    pred = pred*_out_std + _out_mean
    Phi = 0.5*torch.sum((y - pred[0, 0])**2)/sigma**2
    Phi.backward()
    return Phi.item(), u_t.grad.detach().cpu().numpy()


# --- proposals ---
def propose_pcn(u, delta, prior_sample):
    """Vanilla pCN. Prior-reversible => acceptance is just exp(Phi(u)-Phi(u*))."""
    xi = prior_sample()
    return np.sqrt(1 - 2*delta)*u + np.sqrt(2*delta)*xi

def pcn_mala_coeffs(delta):
    """Crank-Nicolson (dimension-robust) coefficients for the Langevin proposal."""
    denom = 1 + delta/2
    rho = (1 - delta/2)/denom        # state-retention
    c1  = delta/denom                # drift weight
    c2  = np.sqrt(2*delta)/denom     # noise weight
    return rho, c1, c2


In [ ]:
# ===========================================================================
# MCMC SAMPLER  (vanilla pCN  <->  pCN-MALA, switched by USE_GRADIENT_PROPOSAL)
# ===========================================================================
from collections import deque

# Build the prior at the working resolution, scaled to match sample_GP_operator.
_target_std = np.mean([sample_GP_operator(N=MCMC_resolution).std() for _ in range(50)])
prior_sample, cinv_quad, apply_C = make_gp_prior(MCMC_resolution, target_std=_target_std)

# --- state ---
delta   = MCMC_delta
u       = prior_sample()
Phi_cur = Phi_hard(u, true_observations, sigma)          # target potential (hard model)
g_cur   = Phi_soft_and_grad(u, true_observations, sigma)[1] if USE_GRADIENT_PROPOSAL else None

# --- bookkeeping (names kept compatible with the plotting cells below) ---
theta_evolution = [T_push(u, tau=-1)]
loss_evolution  = [-Phi_cur]
accepted        = []
nll_window      = deque(maxlen=200)
n_acc_window    = 0
adapting        = True

best_Phi   = Phi_cur
best_theta = theta_evolution[0].copy()
post_sum   = np.zeros_like(u)        # running sum of the {3,12} field, for the posterior mean
post_count = 0

for i in range(MCMC_n_iterations):

    if not USE_GRADIENT_PROPOSAL:
        # ---------- vanilla pCN ----------
        u_star   = propose_pcn(u, delta, prior_sample)
        Phi_star = Phi_hard(u_star, true_observations, sigma)
        log_alpha = Phi_cur - Phi_star                    # prior terms cancel (reversible)
        g_star = None

    else:
        # ---------- pCN-MALA ----------
        rho, c1, c2 = pcn_mala_coeffs(delta)
        mean_fwd = rho*u - c1*apply_C(g_cur)              # drift uses preconditioned gradient
        xi       = prior_sample()
        u_star   = mean_fwd + c2*xi

        Phi_star          = Phi_hard(u_star, true_observations, sigma)   # accept against hard model
        _, g_star         = Phi_soft_and_grad(u_star, true_observations, sigma)
        mean_rev          = rho*u_star - c1*apply_C(g_star)

        # full Metropolis-Hastings: log[pi(u*)/pi(u)] + log[q(u*->u)/q(u->u*)]
        d_log_pi = (Phi_cur - Phi_star) - 0.5*(cinv_quad(u_star) - cinv_quad(u))
        d_log_q  = -(1.0/(2*c2**2))*(cinv_quad(u - mean_rev) - cinv_quad(u_star - mean_fwd))
        log_alpha = d_log_pi + d_log_q

    # ---------- accept / reject ----------
    if np.log(np.random.rand() + 1e-300) < log_alpha:
        u, Phi_cur = u_star, Phi_star
        if USE_GRADIENT_PROPOSAL:
            g_cur = g_star
        accepted.append(1); n_acc_window += 1
    else:
        accepted.append(0)

    theta_cur = T_push(u, tau=-1)
    theta_evolution.append(theta_cur)
    loss_evolution.append(-Phi_cur)
    nll_window.append(Phi_cur)

    if Phi_cur < best_Phi:
        best_Phi, best_theta = Phi_cur, theta_cur.copy()
    if i >= burn_in:
        post_sum += theta_cur; post_count += 1

    # ---------- adapt delta during burn-in ----------
    if (i + 1) % adapt_interval == 0:
        acc_rate = n_acc_window/adapt_interval
        n_acc_window = 0
        if adapting:
            delta = float(np.clip(delta*np.exp(acc_rate - target_acceptance), delta_min, delta_max))
            if (i + 1) >= adapt_cutoff:
                adapting = False
                print(f">>> Adaptation frozen at delta={delta:.6f}")
        print(f"Iter {i+1}/{MCMC_n_iterations} | accept={acc_rate:.3f} | "
              f"delta={delta:.6f} | adapting={adapting} | "
              f"avg NLL(200)={np.mean(nll_window):.3f} | best NLL={best_Phi:.3f}")

# --- posterior summaries (use these instead of the last sample) ---
MCMC_delta = delta                                   # expose adapted value
ll_current = -Phi_cur
posterior_mean_field = post_sum/max(post_count, 1)   # continuous in [3,12]: prob-weighted media
print(f"\nDone. best NLL={best_Phi:.3f} | final NLL={Phi_cur:.3f} | "
      f"true-coeff NLL (floor)={-log_likelihood(theta_true_tensor, true_observations, model, sigma=sigma):.3f}")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=300)
ax.plot(-np.array(loss_evolution))
ax.set_yscale('log')
ax.set_title("Negative Log-likelihood vs Iteration", fontsize=11, loc='center', pad=6)
ax.set_xlabel("Iteration", fontsize=9)
ax.set_ylabel("NegativeLog-likelihood (log scale)", fontsize=9)
ax.tick_params(labelsize=7)
plt.tight_layout()
# plt.savefig('figures_for_report/loss_curve.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
with torch.no_grad():
    forwards = model(true_field[0,0].unsqueeze(0).unsqueeze(0).to(device))

# Compute predicted solution
average_predicted_field = T_push(torch.from_numpy(np.mean(theta_evolution[-10000::1000], axis=0)), tau=-1, boundary=7.5).unsqueeze(0).unsqueeze(0)
print(average_predicted_field.shape)
with torch.no_grad():
    predicted_solution = model(average_predicted_field.to(device)).cpu().numpy()[0, 0]

average_predicted_field = average_predicted_field.squeeze(0).squeeze(0)

# FNO forward pass on true field
fno_prediction = forwards.cpu().numpy()[0, 0]

# ----------------------
# First figure (top row)
# ----------------------
fig1, axs1 = plt.subplots(1, 2, figsize=(12, 5), dpi=300, constrained_layout=True)
fields1 = [
    true_field.cpu().numpy()[0, 0],
    average_predicted_field,
]
titles1 = [
    "True Diffusion Coefficient",
    "Inferred Diffusion Coefficient",
]
for ax, field, title in zip(axs1, fields1, titles1):
    im = ax.imshow(field, extent=[0, 1, 0, 1], origin='lower')
    ax.set_title(title, fontsize=14, loc='center', pad=6)
    ax.tick_params(labelsize=9)

fig1.colorbar(im, ax=axs1, fraction=0.03, pad=0.02, shrink=1.0, aspect=20)
# plt.savefig('figures_for_report/single_true_vs_inferred_coefficients.png', bbox_inches='tight', dpi=300)
plt.show()

# ----------------------
# Second figure (bottom row)
# ----------------------
fig2, axs2 = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

fields2 = [
    true_observations.cpu().numpy()[0, 0],
    fno_prediction,
    predicted_solution,
]

titles2 = [
    "True Flow Field",
    "FNO Prediction (True Coefficient)",
    "Inferred Flow Field under FNO",
]

from matplotlib.colors import Normalize

for ax, field, title in zip(axs2, fields2, titles2):
    field_norm = (field - field.min()) / (field.max() - field.min())
    im = ax.imshow(field_norm, extent=[0, 1, 0, 1], origin='lower', vmin=0, vmax=1)
    ax.set_title(title, fontsize=11, loc='center', pad=6)
    ax.tick_params(labelsize=7)

fig2.colorbar(im, ax=axs2, fraction=0.015, pad=0.04)
# plt.savefig('figures_for_report/single_true_vs_inferred_flows.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, axs = plt.subplots(1, 2, figsize=(12, 5), dpi=300, constrained_layout=True)

# Left: "Inferred Flow Field" from fig2 (normalised)
flow_field = predicted_solution
flow_field_norm = (flow_field - flow_field.min()) / (flow_field.max() - flow_field.min())
im0 = axs[0].imshow(flow_field_norm, extent=[0, 1, 0, 1], origin='lower', vmin=0, vmax=1)
axs[0].set_title("64x64 Inferred Flow Field", fontsize=14, loc='center', pad=6)
axs[0].tick_params(labelsize=9)
divider0 = make_axes_locatable(axs[0])
cax0 = divider0.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im0, cax=cax0)

# Right: "Inferred Diffusion Coefficient" from fig1
diff_field = average_predicted_field
im1 = axs[1].imshow(diff_field, extent=[0, 1, 0, 1], origin='lower')
axs[1].set_title("64x64 Inferred Diffusion Coefficient", fontsize=14, loc='center', pad=6)
axs[1].tick_params(labelsize=9)
divider1 = make_axes_locatable(axs[1])
cax1 = divider1.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im1, cax=cax1)

plt.savefig('figures_for_report/super_res_combined_inferred.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# FNO error on the true coefficient -- this is your best possible NLL
true_field = dataset.test_dbs[64].x[0, 0].unsqueeze(0).unsqueeze(0)  # super-resolution task, so get the 64x64 true field
theta_true_tensor = torch.tensor(true_field.cpu().numpy()[0, 0], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
ll_true = log_likelihood(theta_true_tensor, true_observations, model, sigma=sigma)
print(f"NLL with true coefficient: {ll_true:.2f}")
print(f"NLL of MCMC solution:      {ll_current:.2f}")

In [ ]:
# MCMC solution
pred = torch.tensor(predicted_solution).unsqueeze(0).unsqueeze(0)
pred_norm = (pred - pred.min()) / (pred.max() - pred.min())

obs = true_observations.cpu()[0, 0]
obs_norm = (obs - obs.min()) / (obs.max() - obs.min())
obs_norm = obs_norm.unsqueeze(0).unsqueeze(0)

h1_pred = h1loss(pred_norm, obs_norm)
l2_pred = l2loss(pred_norm, obs_norm)
print(f"H1 Loss of MCMC solution: {h1_pred.item():.4f}")
print(f"L2 Loss of MCMC solution: {l2_pred.item():.4f}")

# True coefficient baseline
with torch.no_grad():
    FNO_predicted_true = model(theta_true_tensor).cpu()[0, 0]

fno_norm = (FNO_predicted_true - FNO_predicted_true.min()) / (FNO_predicted_true.max() - FNO_predicted_true.min())
fno_norm = fno_norm.unsqueeze(0).unsqueeze(0)

h1_true = h1loss(fno_norm, obs_norm)
l2_true = l2loss(fno_norm, obs_norm)
print(f"H1 Loss of true solution: {h1_true.item():.4f}")
print(f"L2 Loss of true solution: {l2_true.item():.4f}")

In [ ]:
# Estimate a reasonable sigma from the FNO's own error on the true field
with torch.no_grad():
    fno_pred = model(theta_true_tensor)
    # unnormalise...
    residuals = (true_observations - fno_pred[0,0])
    print(f"Suggested sigma: {residuals.std().item():.4f}")

In [ ]:
# Plot how much the chain moves each step
diffs = [np.mean(np.abs(theta_evolution[i] - theta_evolution[i-1])) 
         for i in range(1, len(theta_evolution))]
plt.plot(diffs)
plt.title("Mean absolute change per step")